# Student Persistence Model
## 02. Early-Warning Features

This notebook builds features from the first 28 days of each course, so the model only uses information an adviser would have by week four.

## Setup

In [1]:
import pandas as pd

DATA_DIR = "data"
CUTOFF_DAY = 28  # week four of the course
KEYS = ["id_student", "code_module", "code_presentation"]  # one enrollment

## Load and clean student information

The same cleaning steps from `01_explore.ipynb` are repeated here so this notebook runs on its own.

In [2]:
info = pd.read_csv(f"{DATA_DIR}/studentInfo.csv")
info["imd_band"] = info["imd_band"].replace({"10-20": "10-20%", "?": "Unknown"})
print(info.shape)

(32593, 12)


## Load registration information

Missing values in this dataset are recorded as `?`, so they are read in as empty values.

In [3]:
reg = pd.read_csv(f"{DATA_DIR}/studentRegistration.csv", na_values="?")
print(reg.shape)
reg.head()

(32593, 5)


,code_module,code_presentation,id_student,date_registration,date_unregistration
0,AAA,2013J,11391,-159.0,NaN
1,AAA,2013J,28400,-53.0,NaN
2,AAA,2013J,30268,-92.0,12.0
3,AAA,2013J,31604,-52.0,NaN
4,AAA,2013J,32885,-176.0,NaN


## When did students unregister?

`date_unregistration` is the day a student left, counted from the course start. Negative values mean they left before the course began, and empty values mean they never unregistered.

In [4]:
reg["date_unregistration"].describe()

count    10072.000000
mean        49.757645
std         82.460890
min       -365.000000
25%         -2.000000
50%         27.000000
75%        109.000000
max        444.000000
Name: date_unregistration, dtype: float64

## Combine student and registration information

In [5]:
df = info.merge(reg, on=KEYS, how="left", validate="one_to_one")
print(df.shape)

(32593, 14)


### Data quality check

Enrollments marked `Withdrawn` should normally have an unregistration date. This counts any that do not.

In [6]:
no_date = ((df["final_result"] == "Withdrawn") & df["date_unregistration"].isna()).sum()
print(f"Withdrawn enrollments with no unregistration date: {no_date}")

Withdrawn enrollments with no unregistration date: 93


## Remove students who left by the cutoff

These students had already withdrawn by week four, so predicting for them would mean predicting the past (data leakage).

In [7]:
left_early = df["date_unregistration"] <= CUTOFF_DAY
print(f"Enrollments that ended by day {CUTOFF_DAY}: {left_early.sum()}")

df = df[~left_early].copy()
print(f"Enrollments remaining: {len(df)}")

Enrollments that ended by day 28: 5055
Enrollments remaining: 27538


## Define the outcome

`withdrew` is 1 if the enrollment ended in withdrawal after the cutoff, and 0 otherwise.

In [8]:
df["withdrew"] = (df["final_result"] == "Withdrawn").astype(int)
print(f"Withdrawal rate after removing early leavers: {df['withdrew'].mean():.3f}")

Withdrawal rate after removing early leavers: 0.186


## Course activity in the first four weeks

`studentVle.csv` records how many times each student clicked on course materials each day. Only activity before the cutoff is used. Day 0 is the course start, and negative days are activity before the course began.

In [9]:
vle = pd.read_csv(f"{DATA_DIR}/studentVle.csv", usecols=KEYS + ["date", "sum_click"])
vle = vle[vle["date"] < CUTOFF_DAY]
print(vle.shape)

(2801311, 5)


In [10]:
in_course = vle[vle["date"] >= 0]
groups = in_course.groupby(KEYS)

activity = pd.DataFrame({
    "total_clicks": groups["sum_click"].sum(),
    "active_days": groups["date"].nunique(),
    "last_active_day": groups["date"].max(),
})

activity["clicks_week1"] = in_course[in_course["date"] < 7].groupby(KEYS)["sum_click"].sum()
activity["clicks_week4"] = in_course[in_course["date"] >= 21].groupby(KEYS)["sum_click"].sum()
activity["pre_course_clicks"] = vle[vle["date"] < 0].groupby(KEYS)["sum_click"].sum()

activity = activity.reset_index()
activity.head()

,id_student,code_module,code_presentation,total_clicks,active_days,last_active_day,clicks_week1,clicks_week4,pre_course_clicks
0,6516,AAA,2014J,543,18,27,229.0,193.0,256.0
1,8462,DDD,2013J,259,14,25,81.0,23.0,81.0
2,8462,DDD,2014J,10,1,10,NaN,NaN,NaN
3,11391,AAA,2013J,303,7,18,183.0,NaN,98.0
4,23629,BBB,2013B,39,5,21,23.0,2.0,14.0


## Combine activity with student information

Students with no recorded activity get zero clicks. `days_since_last_active` measures how long a student had been inactive by the cutoff.

In [11]:
df = df.merge(activity, on=KEYS, how="left", validate="one_to_one")

click_cols = ["total_clicks", "active_days", "clicks_week1", "clicks_week4", "pre_course_clicks"]
df[click_cols] = df[click_cols].fillna(0)

# Never active in the course window: inactive for the whole period
df["days_since_last_active"] = (CUTOFF_DAY - 1 - df["last_active_day"]).fillna(CUTOFF_DAY)
df = df.drop(columns="last_active_day")

# How many days before the course start the student registered
df["days_registered_early"] = -df["date_registration"]

print(df.shape)

(27538, 22)


### Check the activity features

In [12]:
print(f"Enrollments with no activity in the first four weeks: {(df['total_clicks'] == 0).sum()}")
df.groupby("withdrew")[click_cols + ["days_since_last_active"]].median()

Enrollments with no activity in the first four weeks: 1246


,total_clicks,active_days,clicks_week1,clicks_week4,pre_course_clicks,days_since_last_active
withdrew,,,,,,
0,184.0,10.0,37.0,27.0,29.0,2.0
1,141.0,8.0,23.0,16.0,19.0,2.0


## Save the feature table

The sensitive columns (gender, age, disability, IMD band, region) are kept in the file so they can be used to check fairness later. The file is saved in `data/`, so it is not uploaded to GitHub.

In [13]:
feature_cols = [
    "total_clicks", "active_days", "clicks_week1", "clicks_week4",
    "pre_course_clicks", "days_since_last_active", "days_registered_early",
    "num_of_prev_attempts", "studied_credits", "highest_education",
]
fairness_cols = ["gender", "age_band", "disability", "imd_band", "region"]

final = df[KEYS + feature_cols + fairness_cols + ["withdrew"]]
final.to_csv(f"{DATA_DIR}/features.csv", index=False)
print(final.shape)
print(final.isna().sum()[final.isna().sum() > 0])

(27538, 19)
days_registered_early    7
dtype: int64


## Summary

- Enrollments removed because the student left by day 28: 5055
- Enrollments remaining: 27538
- Withdrawal rate after removing early leavers: 0.186
- What the activity medians show about students who later withdrew: Students who later withdrew were less active from the start. Their median clicks were about 23% lower over the first four weeks (141 vs. 184) and about 40% lower in both week 1 (23 vs. 37) and week 4 (16 vs. 27). They also clicked less before the course began (19 vs. 29). Days since last activity was the same for both groups (2 days), so recent inactivity alone does not separate them. These are group medians, and the two groups likely overlap a lot, so individual predictions will need the model.